# MLP Probing on Mutopia Embeddings

Probe frozen lilyBERT embeddings (layers 3/6/9/12) for composer and style classification using a simple MLP with 5-fold stratified cross-validation.

In [31]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, top_k_accuracy_score

SEED = 42
N_FOLDS = 5
MIN_COUNT = 10
LAYERS = ["layer_3", "layer_6", "layer_9", "layer_12"]
DATA_DIR = Path("../data/mutopia")
DATASET_PATH = DATA_DIR / "dataset_mutopia.json"

## Data Loading

In [32]:
with open(DATASET_PATH) as f:
    dataset = json.load(f)

print(f"Total entries: {len(dataset)}")

# Skip entries without embeddings
dataset = [entry for entry in dataset if entry.get("embeddings") is not None]
print(f"Entries with embeddings: {len(dataset)}")

# Load all embeddings into a dict: layer -> array of shape (n_entries, 768)
embeddings = {}
for layer in LAYERS:
    embs = [np.load(DATA_DIR / entry["embeddings"][layer]) for entry in dataset]
    embeddings[layer] = np.stack(embs)
    print(f"{layer}: {embeddings[layer].shape}")

composers = [entry["composer"] for entry in dataset]
styles = [entry["style"] for entry in dataset]

Total entries: 2123
Entries with embeddings: 1987
layer_3: (1987, 768)
layer_6: (1987, 768)
layer_9: (1987, 768)
layer_12: (1987, 768)


In [33]:
def filter_by_min_count(labels, min_count):
    """Return a boolean mask keeping only labels with >= min_count occurrences."""
    counts = Counter(labels)
    valid = {label for label, count in counts.items() if count >= min_count}
    mask = np.array([label in valid for label in labels])
    kept = {l: c for l, c in counts.items() if l in valid}
    dropped = {l: c for l, c in counts.items() if l not in valid}
    print(f"Kept {len(kept)} classes ({mask.sum()} samples), dropped {len(dropped)} classes ({(~mask).sum()} samples)")
    print(f"Kept: {dict(sorted(kept.items(), key=lambda x: -x[1]))}")
    return mask

## Probing Helper

In [34]:
def run_probing(X, y, task_name, layer_name):
    """Train MLP probe with 5-fold stratified CV. Returns metrics dict."""
    n_classes = len(np.unique(y))
    top_ks = [k for k in [1, 3, 5] if k <= n_classes]

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_metrics = {
        "precision": [],
        "recall": [],
        **{f"top{k}": [] for k in top_ks},
    }

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        clf = MLPClassifier(
            hidden_layer_sizes=(512,),
            max_iter=500,
            random_state=SEED,
        )
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_val)
        y_proba = clf.predict_proba(X_val)

        prec = precision_score(y_val, y_pred, average="macro", zero_division=0)
        rec = recall_score(y_val, y_pred, average="macro", zero_division=0)
        fold_metrics["precision"].append(prec)
        fold_metrics["recall"].append(rec)

        top_k_str_parts = []
        for k in top_ks:
            topk = top_k_accuracy_score(y_val, y_proba, k=k, labels=clf.classes_)
            fold_metrics[f"top{k}"].append(topk)
            top_k_str_parts.append(f"top{k}={topk:.4f}")

        print(f"  Fold {fold_idx + 1}: {' '.join(top_k_str_parts)}  prec={prec:.4f}  rec={rec:.4f}")

    results = {}
    for metric_name, values in fold_metrics.items():
        results[f"{metric_name}_mean"] = np.mean(values)
        results[f"{metric_name}_std"] = np.std(values)

    summary_parts = [
        f"top{k}={results[f'top{k}_mean']:.4f}±{results[f'top{k}_std']:.4f}"
        for k in top_ks
    ]
    summary_parts.append(f"prec={results['precision_mean']:.4f}±{results['precision_std']:.4f}")
    summary_parts.append(f"rec={results['recall_mean']:.4f}±{results['recall_std']:.4f}")

    print(f"  => {task_name} | {layer_name}: {'  '.join(summary_parts)}")

    return results

---
## Composer Classification

In [35]:
composer_mask = filter_by_min_count(composers, MIN_COUNT)
composer_labels = np.array(composers)[composer_mask]

le_composer = LabelEncoder()
y_composer = le_composer.fit_transform(composer_labels)
print(f"\nNumber of classes: {len(le_composer.classes_)}")
print(f"Classes: {list(le_composer.classes_)}")

Kept 30 classes (1438 samples), dropped 284 classes (549 samples)
Kept: {'BachJS': 371, 'Traditional': 125, 'GiulianiM': 97, 'MozartWA': 93, 'SorF': 73, 'BeethovenLv': 65, 'HoretzkyF': 60, 'HandelGF': 57, 'SchubertF': 42, 'ChopinFF': 42, 'CarcassiM': 39, 'DiabelliA': 36, 'SchumannR': 34, 'CzernyC': 29, 'MontePd': 29, 'Anonymous': 25, 'AguadoD': 23, 'TitelouzeJ': 22, 'HaydnFJ': 21, 'Mendelssohn-BartholdyF': 20, 'BurgmullerJFF': 19, 'KnjzeF': 18, 'VerdiG': 17, 'JoplinS': 16, 'SatieE': 13, 'MonteverdiC': 11, 'BrahmsJ': 11, 'VivaldiA': 10, 'GriegE': 10, 'FaureG': 10}

Number of classes: 30
Classes: [np.str_('AguadoD'), np.str_('Anonymous'), np.str_('BachJS'), np.str_('BeethovenLv'), np.str_('BrahmsJ'), np.str_('BurgmullerJFF'), np.str_('CarcassiM'), np.str_('ChopinFF'), np.str_('CzernyC'), np.str_('DiabelliA'), np.str_('FaureG'), np.str_('GiulianiM'), np.str_('GriegE'), np.str_('HandelGF'), np.str_('HaydnFJ'), np.str_('HoretzkyF'), np.str_('JoplinS'), np.str_('KnjzeF'), np.str_('Mendelssoh

### Layer 3

In [36]:
composer_results_l3 = run_probing(embeddings["layer_3"][composer_mask], y_composer, "Composer", "Layer 3")

  Fold 1: top1=0.4236 top3=0.6458 top5=0.7535  prec=0.2095  rec=0.1952
  Fold 2: top1=0.4028 top3=0.6076 top5=0.7326  prec=0.2253  rec=0.1773
  Fold 3: top1=0.3854 top3=0.5972 top5=0.7188  prec=0.1772  rec=0.1520
  Fold 4: top1=0.4007 top3=0.5889 top5=0.6934  prec=0.1966  rec=0.1693
  Fold 5: top1=0.3449 top3=0.5749 top5=0.6934  prec=0.1779  rec=0.1570
  => Composer | Layer 3: top1=0.3915±0.0263  top3=0.6029±0.0240  top5=0.7183±0.0232  prec=0.1973±0.0185  rec=0.1702±0.0154


### Layer 6

In [37]:
composer_results_l6 = run_probing(embeddings["layer_6"][composer_mask], y_composer, "Composer", "Layer 6")

  Fold 1: top1=0.3507 top3=0.6250 top5=0.7153  prec=0.1951  rec=0.1737
  Fold 2: top1=0.3368 top3=0.5868 top5=0.6910  prec=0.1864  rec=0.1377
  Fold 3: top1=0.3229 top3=0.5521 top5=0.6736  prec=0.1645  rec=0.1120
  Fold 4: top1=0.3554 top3=0.5645 top5=0.6725  prec=0.1980  rec=0.1522
  Fold 5: top1=0.3345 top3=0.5505 top5=0.6620  prec=0.2076  rec=0.1695
  => Composer | Layer 6: top1=0.3401±0.0117  top3=0.5758±0.0278  top5=0.6829±0.0187  prec=0.1903±0.0146  rec=0.1490±0.0225


### Layer 9

In [38]:
composer_results_l9 = run_probing(embeddings["layer_9"][composer_mask], y_composer, "Composer", "Layer 9")

  Fold 1: top1=0.3576 top3=0.5972 top5=0.6944  prec=0.1955  rec=0.1795
  Fold 2: top1=0.3333 top3=0.5868 top5=0.6806  prec=0.1326  rec=0.1250
  Fold 3: top1=0.3229 top3=0.5243 top5=0.6354  prec=0.1585  rec=0.1196
  Fold 4: top1=0.3589 top3=0.5540 top5=0.6829  prec=0.1578  rec=0.1521
  Fold 5: top1=0.3136 top3=0.4913 top5=0.6655  prec=0.1414  rec=0.1402
  => Composer | Layer 9: top1=0.3373±0.0182  top3=0.5507±0.0393  top5=0.6718±0.0204  prec=0.1571±0.0216  rec=0.1433±0.0214


### Layer 12

In [39]:
composer_results_l12 = run_probing(embeddings["layer_12"][composer_mask], y_composer, "Composer", "Layer 12")

  Fold 1: top1=0.4062 top3=0.6424 top5=0.7431  prec=0.2366  rec=0.2040
  Fold 2: top1=0.3715 top3=0.5833 top5=0.7049  prec=0.2325  rec=0.1701
  Fold 3: top1=0.3646 top3=0.5868 top5=0.7083  prec=0.2107  rec=0.1394
  Fold 4: top1=0.4077 top3=0.5854 top5=0.7143  prec=0.2197  rec=0.1834
  Fold 5: top1=0.3554 top3=0.5645 top5=0.7038  prec=0.1888  rec=0.1702
  => Composer | Layer 12: top1=0.3811±0.0217  top3=0.5925±0.0262  top5=0.7149±0.0146  prec=0.2177±0.0171  rec=0.1734±0.0210


---
## Style Classification

In [40]:
style_mask = filter_by_min_count(styles, MIN_COUNT)
style_labels = np.array(styles)[style_mask]

le_style = LabelEncoder()
y_style = le_style.fit_transform(style_labels)
print(f"\nNumber of classes: {len(le_style.classes_)}")
print(f"Classes: {list(le_style.classes_)}")

Kept 10 classes (1972 samples), dropped 3 classes (15 samples)
Kept: {'Baroque': 621, 'Classical': 585, 'Romantic': 400, 'Folk': 110, 'Hymn': 108, 'Renaissance': 57, 'Song': 28, 'Jazz': 22, 'Technique': 21, 'Modern': 20}

Number of classes: 10
Classes: [np.str_('Baroque'), np.str_('Classical'), np.str_('Folk'), np.str_('Hymn'), np.str_('Jazz'), np.str_('Modern'), np.str_('Renaissance'), np.str_('Romantic'), np.str_('Song'), np.str_('Technique')]


### Layer 3

In [41]:
style_results_l3 = run_probing(embeddings["layer_3"][style_mask], y_style, "Style", "Layer 3")

  Fold 1: top1=0.3418 top3=0.8329 top5=0.9468  prec=0.2348  rec=0.1309
  Fold 2: top1=0.4380 top3=0.8532 top5=0.9468  prec=0.2016  rec=0.1921
  Fold 3: top1=0.4315 top3=0.8655 top5=0.9518  prec=0.2997  rec=0.1937
  Fold 4: top1=0.4213 top3=0.8503 top5=0.9518  prec=0.3026  rec=0.1895
  Fold 5: top1=0.4137 top3=0.8325 top5=0.9416  prec=0.2017  rec=0.2115
  => Style | Layer 3: top1=0.4092±0.0347  top3=0.8469±0.0126  top5=0.9478±0.0038  prec=0.2481±0.0450  rec=0.1835±0.0274


### Layer 6

In [42]:
style_results_l6 = run_probing(embeddings["layer_6"][style_mask], y_style, "Style", "Layer 6")

  Fold 1: top1=0.3823 top3=0.8405 top5=0.9443  prec=0.1536  rec=0.1805
  Fold 2: top1=0.4228 top3=0.8304 top5=0.9443  prec=0.1986  rec=0.1820
  Fold 3: top1=0.4112 top3=0.8325 top5=0.9569  prec=0.1961  rec=0.1821
  Fold 4: top1=0.3528 top3=0.8376 top5=0.9492  prec=0.1693  rec=0.1317
  Fold 5: top1=0.4112 top3=0.8477 top5=0.9442  prec=0.2126  rec=0.1659
  => Style | Layer 6: top1=0.3960±0.0254  top3=0.8377±0.0061  top5=0.9478±0.0049  prec=0.1860±0.0214  rec=0.1684±0.0193


### Layer 9

In [43]:
style_results_l9 = run_probing(embeddings["layer_9"][style_mask], y_style, "Style", "Layer 9")

  Fold 1: top1=0.4051 top3=0.8304 top5=0.9443  prec=0.3083  rec=0.1756
  Fold 2: top1=0.4278 top3=0.8405 top5=0.9468  prec=0.2871  rec=0.1911
  Fold 3: top1=0.3909 top3=0.8325 top5=0.9518  prec=0.1693  rec=0.1773
  Fold 4: top1=0.4340 top3=0.8376 top5=0.9467  prec=0.2233  rec=0.1882
  Fold 5: top1=0.4137 top3=0.8452 top5=0.9467  prec=0.2219  rec=0.1486
  => Style | Layer 9: top1=0.4143±0.0155  top3=0.8372±0.0054  top5=0.9473±0.0024  prec=0.2420±0.0499  rec=0.1761±0.0150


### Layer 12

In [44]:
style_results_l12 = run_probing(embeddings["layer_12"][style_mask], y_style, "Style", "Layer 12")

  Fold 1: top1=0.4253 top3=0.8633 top5=0.9494  prec=0.4197  rec=0.2037
  Fold 2: top1=0.3544 top3=0.8405 top5=0.9519  prec=0.3076  rec=0.1581
  Fold 3: top1=0.3858 top3=0.8528 top5=0.9645  prec=0.1860  rec=0.1645
  Fold 4: top1=0.4442 top3=0.8350 top5=0.9442  prec=0.2722  rec=0.2186
  Fold 5: top1=0.4188 top3=0.8426 top5=0.9467  prec=0.2051  rec=0.1992
  => Style | Layer 12: top1=0.4057±0.0318  top3=0.8469±0.0100  top5=0.9513±0.0071  prec=0.2781±0.0834  rec=0.1888±0.0235


---
## Summary

In [47]:
def fmt(mean, std):
    return f"{mean:.3f} ± {std:.3f}"

composer_results = {
    "Layer 3": composer_results_l3,
    "Layer 6": composer_results_l6,
    "Layer 9": composer_results_l9,
    "Layer 12": composer_results_l12,
}
style_results = {
    "Layer 3": style_results_l3,
    "Layer 6": style_results_l6,
    "Layer 9": style_results_l9,
    "Layer 12": style_results_l12,
}

rows = []
for layer in ["Layer 3", "Layer 6", "Layer 9", "Layer 12"]:
    c = composer_results[layer]
    s = style_results[layer]
    rows.append({
        "Layer": layer,
        "Composer Top-1": fmt(c["top1_mean"], c["top1_std"]),
        "Composer Top-3": fmt(c["top3_mean"], c["top3_std"]),
        "Composer Precision": fmt(c["precision_mean"], c["precision_std"]),
        "Composer Recall": fmt(c["recall_mean"], c["recall_std"]),
        "Style Top-1": fmt(s["top1_mean"], s["top1_std"]),
        "Style Top-3": fmt(s["top3_mean"], s["top3_std"]),
        "Style Precision": fmt(s["precision_mean"], s["precision_std"]),
        "Style Recall": fmt(s["recall_mean"], s["recall_std"]),
    })

summary_df = pd.DataFrame(rows).set_index("Layer")
display(summary_df)

,Composer Top-1,Composer Top-3,Composer Precision,Composer Recall,Style Top-1,Style Top-3,Style Precision,Style Recall
Layer,,,,,,,,
Layer 3,0.391 ± 0.026,0.603 ± 0.024,0.197 ± 0.019,0.170 ± 0.015,0.409 ± 0.035,0.847 ± 0.013,0.248 ± 0.045,0.184 ± 0.027
Layer 6,0.340 ± 0.012,0.576 ± 0.028,0.190 ± 0.015,0.149 ± 0.023,0.396 ± 0.025,0.838 ± 0.006,0.186 ± 0.021,0.168 ± 0.019
Layer 9,0.337 ± 0.018,0.551 ± 0.039,0.157 ± 0.022,0.143 ± 0.021,0.414 ± 0.016,0.837 ± 0.005,0.242 ± 0.050,0.176 ± 0.015
Layer 12,0.381 ± 0.022,0.592 ± 0.026,0.218 ± 0.017,0.173 ± 0.021,0.406 ± 0.032,0.847 ± 0.010,0.278 ± 0.083,0.189 ± 0.023
